# DS102 — thí nghiệm trên hosted Colab, không dùng pretrained
Chỉ huấn luyện từ bốn CSV cuộc thi; giữ dev riêng. Máy cá nhân chỉ dùng cầu nối MCP và chuẩn bị file.
Run All để chạy CV, so sánh với v3 và chỉ tạo submission_v4.csv nếu mô hình kết hợp vượt ngưỡng CV đã đặt. Chạy CPU Colab có thể mất nhiều phút. Kết quả CV được dùng chọn mô hình; điểm Kaggle mới chỉ biết sau khi nộp.


In [ ]:
# DS102: kiểm tra runtime qua MCP, không huấn luyện
import os, platform, json, shutil, subprocess
from pathlib import Path

info = {
    'system': platform.system(),
    'python': platform.python_version(),
    'cpu_count': os.cpu_count(),
    'colab_release': os.environ.get('COLAB_RELEASE_TAG'),
    'content_directory': Path('/content').is_dir(),
}
if Path('/proc/meminfo').is_file():
    total = next(line for line in Path('/proc/meminfo').read_text().splitlines() if line.startswith('MemTotal:'))
    info['ram_gib'] = round(int(total.split()[1]) / 1024**2, 2)
if shutil.which('nvidia-smi'):
    p = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True, timeout=10)
    info['gpu'] = p.stdout.strip() if p.returncode == 0 else 'GPU query unavailable'
else:
    info['gpu'] = 'No NVIDIA GPU detected'
info['hosted_colab_indicators'] = bool(info['system'] == 'Linux' and info['colab_release'] and info['content_directory'])
print(json.dumps(info, ensure_ascii=False, indent=2))
print('Diagnostic only: no training started.')

In [ ]:
# Tải dữ liệu cuộc thi lên Colab; không dùng dữ liệu ngoài
import os, platform, zipfile, io
from pathlib import Path
from google.colab import files
if platform.system() != 'Linux' or not os.environ.get('COLAB_RELEASE_TAG'):
    raise RuntimeError('Chỉ tiếp tục trên hosted Colab, không dùng local runtime.')
DATA_DIR = Path('/content/ds102_data')
DATA_DIR.mkdir(exist_ok=True)
required = ['ds102_train.csv','ds102_dev.csv','ds102_test.csv','ds102_samplesubmission.csv']
if not all((DATA_DIR / n).is_file() for n in required):
    print('Chọn colab_data.zip đã chuẩn bị hoặc chọn đủ bốn CSV.')
    uploaded = files.upload()
    for name, payload in uploaded.items():
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(payload)) as z:
                for member in z.namelist():
                    basename = Path(member).name
                    if basename in required:
                        (DATA_DIR / basename).write_bytes(z.read(member))
        elif Path(name).name in required:
            (DATA_DIR / Path(name).name).write_bytes(payload)
    del uploaded
missing = [n for n in required if not (DATA_DIR / n).is_file()]
if missing:
    raise FileNotFoundError(f'Còn thiếu: {missing}; chạy lại ô để tải bổ sung.')
print('READY: đủ bốn file trong', DATA_DIR)


In [ ]:
import os, sys, json, subprocess, importlib.metadata
from pathlib import Path
if not os.environ.get('COLAB_RELEASE_TAG'):
    raise RuntimeError('Hosted Colab required')
if importlib.metadata.version('scikit-learn') != '1.7.2':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','scikit-learn==1.7.2'])
source_files = {"train_submit.py":"\"\"\"CPU baseline cho DS102: TF-IDF + Linear SVM, chọn theo F1-macro.\n\nChỉ dùng comment và nhãn train. Dev được đánh giá sau khi chọn tham số\nbằng cross-validation trên train; không gộp dev vào train.\n\"\"\"\n\nimport argparse\nimport html\nimport json\nimport platform\nimport re\nimport unicodedata\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom scipy import sparse\nfrom sklearn.base import BaseEstimator, TransformerMixin\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.pipeline import FeatureUnion, Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import LinearSVC\n\n\nSEED = 102\nLABELS = [0, 1, 2, 3]\nFILES = {name: f\"ds102_{name}.csv\" for name in [\"train\", \"dev\", \"test\", \"samplesubmission\"]}\n\n\ndef normalize_text(value):\n    \"\"\"Giữ dấu, emoji, dấu câu và ký tự lặp vì có thể giúp nhận diện spam.\"\"\"\n    text = html.unescape(unicodedata.normalize(\"NFC\", str(value)))\n    text = text.replace(\"\\\\r\\\\n\", \"\\n\").replace(\"\\\\n\", \"\\n\").replace(\"\\\\r\", \"\\n\")\n    return re.sub(r\"\\s+\", \" \", text).strip().lower()\n\n\nclass ReviewStats(BaseEstimator, TransformerMixin):\n    \"\"\"Đặc trưng độ dài, lặp ký tự và tỷ lệ ký tự; không có từ khóa nhãn.\"\"\"\n\n    def fit(self, X, y=None):\n        return self\n\n    def transform(self, X):\n        rows = []\n        for text in X:\n            chars = len(text)\n            words = text.split()\n            nwords = len(words)\n            denom = max(chars, 1)\n            word_denom = max(nwords, 1)\n            letters = [c for c in text if c.isalpha()]\n            longest_run = max((len(m.group()) for m in re.finditer(r\"(.)\\1*\", text)), default=0)\n            rows.append([\n                np.log1p(chars), np.log1p(nwords), chars / word_denom,\n                len(set(words)) / word_denom, len(set(text)) / denom,\n                longest_run / denom, np.log1p(longest_run),\n                sum(c.isdigit() for c in text) / denom,\n                len(letters) / denom,\n                sum(not c.isalnum() and not c.isspace() for c in text) / denom,\n                sum(ord(c) > 0xFFFF for c in text) / denom,\n                text.count(\"!\") / denom, text.count(\"?\") / denom,\n                float(bool(re.search(r\"https?://|www\\.\", text))),\n            ])\n        return sparse.csr_matrix(np.asarray(rows, dtype=np.float32))\n\n\ndef make_features():\n    return FeatureUnion([\n        (\"word\", TfidfVectorizer(\n            ngram_range=(1, 2), min_df=2, max_df=1.0, max_features=90000,\n            sublinear_tf=True, lowercase=False, dtype=np.float32,\n            token_pattern=r\"(?u)\\b\\w+\\b\",\n        )),\n        (\"char\", TfidfVectorizer(\n            analyzer=\"char\", ngram_range=(2, 5), min_df=3, max_features=160000,\n            sublinear_tf=True, lowercase=False, dtype=np.float32,\n        )),\n        (\"stats\", Pipeline([\n            (\"extract\", ReviewStats()), (\"scale\", StandardScaler(with_mean=False)),\n        ])),\n    ], transformer_weights={\"word\": 1.0, \"char\": 1.0, \"stats\": 0.1})\n\n\ndef make_model(config, y):\n    weight = config[\"weight\"]\n    if weight == \"sqrt_balanced\":\n        counts = pd.Series(y).value_counts()\n        weight = {int(k): float(np.sqrt(len(y) / (len(counts) * v))) for k, v in counts.items()}\n    elif weight == \"none\":\n        weight = None\n    return LinearSVC(C=config[\"C\"], class_weight=weight, dual=True,\n                     max_iter=10000, tol=1e-4, random_state=SEED)\n\n\ndef find_data_dir(supplied=None):\n    if supplied:\n        path = Path(supplied).expanduser().resolve()\n        if all((path / f).is_file() for f in FILES.values()):\n            return path\n        raise FileNotFoundError(f\"Thiếu một trong 4 file CSV tại {path}\")\n    candidates = [Path.cwd(), Path.cwd() / \"data\", Path.cwd() / \"dataset\"]\n    kaggle = Path(\"/kaggle/input\")\n    if kaggle.exists():\n        candidates += sorted({p.parent for p in kaggle.rglob(FILES[\"train\"])})\n    matches = [p for p in candidates if all((p / f).is_file() for f in FILES.values())]\n    if len(matches) == 1:\n        return matches[0].resolve()\n    raise FileNotFoundError(\"Hãy truyền --data-dir đến thư mục chứa đúng 4 CSV; trên Kaggle dùng Add Input trước.\")\n\n\ndef load_data(data_dir):\n    data = {name: pd.read_csv(data_dir / file, keep_default_na=False) for name, file in FILES.items()}\n    for name in [\"train\", \"dev\", \"test\"]:\n        required = {\"comment\", \"spam_label\"} if name != \"test\" else {\"comment\", \"id\"}\n        if not required.issubset(data[name].columns):\n            raise ValueError(f\"{name}: thiếu cột {required - set(data[name].columns)}\")\n        if data[name].empty:\n            raise ValueError(f\"{name} rỗng\")\n    for name in [\"train\", \"dev\"]:\n        values = pd.to_numeric(data[name][\"spam_label\"], errors=\"raise\")\n        if not values.isin(LABELS).all():\n            raise ValueError(f\"{name}: nhãn phải thuộc {LABELS}\")\n        data[name][\"spam_label\"] = values.astype(int)\n    if set(data[\"train\"][\"spam_label\"]) != set(LABELS):\n        raise ValueError(\"Train phải có đủ 4 nhãn\")\n    sample = data[\"samplesubmission\"]\n    if list(sample.columns) != [\"ID\", \"spam_label\"]:\n        raise ValueError(\"Sample phải có hai cột ID,spam_label\")\n    test_ids, sample_ids = data[\"test\"][\"id\"].astype(str), sample[\"ID\"].astype(str)\n    if not test_ids.is_unique or not sample_ids.is_unique or set(test_ids) != set(sample_ids):\n        raise ValueError(\"ID trong test và sample phải duy nhất và trùng tập hợp\")\n    return data\n\n\ndef metrics(y, pred):\n    return {\n        \"macro_f1\": float(f1_score(y, pred, labels=LABELS, average=\"macro\", zero_division=0)),\n        \"accuracy\": float(accuracy_score(y, pred)),\n        \"classification_report\": classification_report(y, pred, labels=LABELS, output_dict=True, zero_division=0),\n        \"confusion_matrix\": confusion_matrix(y, pred, labels=LABELS).tolist(),\n    }\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--data-dir\", default=None)\n    parser.add_argument(\"--output-dir\", default=\"output\")\n    parser.add_argument(\"--folds\", type=int, default=3)\n    args = parser.parse_args(argv)\n    if args.folds < 2:\n        parser.error(\"--folds phải >= 2\")\n    data_dir = find_data_dir(args.data_dir)\n    output = Path(args.output_dir).expanduser().resolve()\n    output.mkdir(parents=True, exist_ok=True)\n    data = load_data(data_dir)\n    train, dev, test = [data[n] for n in [\"train\", \"dev\", \"test\"]]\n    texts = train[\"comment\"].map(normalize_text).to_numpy()\n    dev_texts = dev[\"comment\"].map(normalize_text).to_numpy()\n    test_texts = test[\"comment\"].map(normalize_text).to_numpy()\n    y = train[\"spam_label\"].to_numpy()\n    if train.groupby(train[\"comment\"].map(normalize_text))[\"spam_label\"].first().value_counts().min() < args.folds:\n        raise ValueError(\"Không đủ nhóm cho số folds đã chọn\")\n    print(f\"Train={len(train)}, dev={len(dev)}, test={len(test)}\", flush=True)\n    print(f\"Nhãn train: {train.spam_label.value_counts().sort_index().to_dict()}\", flush=True)\n    configs = [{\"C\": c, \"weight\": weight} for weight in [\"none\", \"sqrt_balanced\", \"balanced\"] for c in [1.0, 4.0]]\n    oof = np.full((len(configs), len(y)), -1, dtype=np.int64)\n    fold_scores = [[] for _ in configs]\n    splitter = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=SEED)\n    for fold, (fit_idx, val_idx) in enumerate(splitter.split(texts, y, groups=texts), 1):\n        print(f\"Fold {fold}/{args.folds}: fit TF-IDF chỉ trên train của fold...\", flush=True)\n        features = make_features()\n        X_fit = features.fit_transform(texts[fit_idx])\n        X_val = features.transform(texts[val_idx])\n        for i, config in enumerate(configs):\n            model = make_model(config, y[fit_idx])\n            model.fit(X_fit, y[fit_idx])\n            pred = model.predict(X_val)\n            oof[i, val_idx] = pred\n            score = f1_score(y[val_idx], pred, labels=LABELS, average=\"macro\", zero_division=0)\n            fold_scores[i].append(float(score))\n            print(f\"  {config}: F1-macro={score:.5f}\", flush=True)\n        del X_fit, X_val, features\n    cv_scores = [float(f1_score(y, p, labels=LABELS, average=\"macro\", zero_division=0)) for p in oof]\n    best_idx = int(np.argmax(cv_scores))\n    best = configs[best_idx]\n    print(f\"Chọn bằng OOF train: {best}, F1-macro={cv_scores[best_idx]:.5f}\", flush=True)\n    print(\"Huấn luyện cấu hình đã chọn trên toàn bộ train...\", flush=True)\n    features = make_features()\n    X_train = features.fit_transform(texts)\n    model = make_model(best, y)\n    model.fit(X_train, y)\n    dev_pred = model.predict(features.transform(dev_texts))\n    dev_y = dev[\"spam_label\"].to_numpy()\n    report = metrics(dev_y, dev_pred)\n    unique_dev = ~pd.Series(dev_texts).isin(set(texts)).to_numpy()\n    print(f\"Dev F1-macro={report['macro_f1']:.5f}, accuracy={report['accuracy']:.5f}\", flush=True)\n    print(classification_report(dev_y, dev_pred, labels=LABELS, digits=4, zero_division=0), flush=True)\n    print(f\"Dev có {int((~unique_dev).sum())} comment chuẩn hóa xuất hiện trong train.\", flush=True)\n    test_pred = model.predict(features.transform(test_texts)).astype(int)\n    by_id = pd.Series(test_pred, index=test[\"id\"].astype(str))\n    submission = data[\"samplesubmission\"].copy()\n    submission[\"spam_label\"] = submission[\"ID\"].astype(str).map(by_id)\n    if submission[\"spam_label\"].isna().any() or not submission[\"spam_label\"].isin(LABELS).all():\n        raise ValueError(\"Submission có nhãn thiếu hoặc không hợp lệ\")\n    submission[\"spam_label\"] = submission[\"spam_label\"].astype(int)\n    submission_path = output / \"submission.csv\"\n    submission.to_csv(submission_path, index=False)\n    saved = pd.read_csv(submission_path)\n    pd.testing.assert_frame_equal(saved, submission, check_dtype=False)\n    if len(saved) != len(test) or not saved[\"ID\"].equals(data[\"samplesubmission\"][\"ID\"]):\n        raise ValueError(\"Sai số hàng hoặc thứ tự ID khi lưu\")\n    detail = pd.DataFrame({\"comment\": dev[\"comment\"], \"true_label\": dev_y, \"predicted_label\": dev_pred,\n                           \"comment_seen_in_train\": ~unique_dev})\n    detail.to_csv(output / \"dev_predictions.csv\", index=False)\n    full_report = {\n        \"seed\": SEED, \"python\": platform.python_version(), \"sklearn\": sklearn.__version__,\n        \"rows\": {k: len(v) for k, v in data.items()}, \"feature_columns\": [\"comment\"],\n        \"selection\": \"StratifiedGroupKFold on train; normalized comments as groups; OOF macro F1\",\n        \"fit_data\": \"train only; dev and test never used to fit TF-IDF, scaler or SVM\",\n        \"candidates\": [{**c, \"fold_macro_f1\": f, \"oof_macro_f1\": s}\n                       for c, f, s in zip(configs, fold_scores, cv_scores)],\n        \"chosen_config\": best, \"dev\": report,\n        \"dev_comments_seen_in_train\": int((~unique_dev).sum()),\n        \"dev_without_comment_overlap\": metrics(dev_y[unique_dev], dev_pred[unique_dev]) if unique_dev.any() else None,\n        \"test_prediction_counts\": {int(k): int(v) for k, v in submission.spam_label.value_counts().items()},\n        \"submission_rows_verified\": len(saved),\n    }\n    (output / \"metrics.json\").write_text(json.dumps(full_report, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    print(f\"Đã kiểm tra và lưu {len(saved)} dự đoán: {submission_path}\", flush=True)\n    return full_report\n\n\nif __name__ == \"__main__\":\n    main()\n","improved_features.py":"\"\"\"Đặc trưng v2: văn bản review và mức trùng với thông tin sản phẩm.\"\"\"\nimport re\nfrom collections import Counter\nimport numpy as np\nfrom scipy import sparse\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.preprocessing import StandardScaler, OneHotEncoder\nfrom train_submit import normalize_text, ReviewStats\n\n\ndef prepare(frame):\n    frame = frame.copy()\n    for col in ['comment', 'product_name', 'description', 'category', 'categories']:\n        frame[col] = frame[col].fillna('').astype(str).map(normalize_text)\n    return frame\n\n\ndef tokens(text):\n    return re.findall(r'(?u)\\b\\w+\\b', text)\n\n\ndef ngrams(words, n):\n    return set(zip(*(words[i:] for i in range(n))))\n\n\ndef dense_features(df):\n    basic = ReviewStats().transform(df.comment).toarray()\n    extra = []\n    for row in df.itertuples(index=False):\n        text = row.comment\n        words = tokens(text)\n        wl = [len(w) for w in words] or [0]\n        wc = Counter(words)\n        n = max(len(words), 1)\n        feats = [np.log1p(max(wl)), np.mean(wl), np.std(wl),\n                 sum(len(w) > 12 for w in words) / n,\n                 max(wc.values(), default=0) / n,\n                 sum(v * (v - 1) for v in wc.values()) / max(n * (n - 1), 1),\n                 text.count('.') / max(len(text), 1),\n                 len(re.findall(r'(.)\\1{3,}', text)) / n,\n                 np.log1p(len(text.split())),\n                 float(row.rating) / 5,\n                 np.log1p(max(float(row.num_sold), 0)),\n                 np.log1p(max(float(row.num_review), 0))]\n        for context in [row.product_name, row.description]:\n            context_words = tokens(context)\n            for size in [1, 2, 3, 4]:\n                a, b = ngrams(words, size), ngrams(context_words, size)\n                overlap = len(a & b)\n                ratio = overlap / max(len(a), 1)\n                feats += [ratio, ratio ** 2, ratio * np.log1p(len(words)), np.log1p(overlap)]\n            feats += [float(len(text) > 10 and text in context), np.log1p(len(context))]\n        extra.append(feats)\n    return np.hstack([basic, np.asarray(extra, dtype=np.float32)])\n\n\nclass FeaturesV2:\n    def __init__(self):\n        self.word = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_features=140000,\n                                   sublinear_tf=True, lowercase=False, dtype=np.float32,\n                                   token_pattern=r'(?u)\\b\\w+\\b')\n        self.char = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), min_df=3,\n                                   max_features=180000, sublinear_tf=True,\n                                   lowercase=False, dtype=np.float32)\n        self.product = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=20000,\n                                      sublinear_tf=True, lowercase=False, dtype=np.float32)\n        self.category = OneHotEncoder(handle_unknown='ignore', dtype=np.float32)\n        self.scale = StandardScaler()\n\n    def fit_transform(self, df):\n        text = sparse.hstack([self.word.fit_transform(df.comment), self.char.fit_transform(df.comment)], format='csr')\n        meta = sparse.hstack([self.product.fit_transform(df.product_name) * .2,\n                             self.category.fit_transform(df[['category', 'rating']].astype(str)) * .2,\n                             sparse.csr_matrix(self.scale.fit_transform(dense_features(df))) * .15], format='csr')\n        return text, meta\n\n    def transform(self, df):\n        text = sparse.hstack([self.word.transform(df.comment), self.char.transform(df.comment)], format='csr')\n        meta = sparse.hstack([self.product.transform(df.product_name) * .2,\n                             self.category.transform(df[['category', 'rating']].astype(str)) * .2,\n                             sparse.csr_matrix(self.scale.transform(dense_features(df))) * .15], format='csr')\n        return text, meta\n\n\ndef nb_ratios(X, y):\n    \"\"\"Log-count ratio one-vs-rest, fit chỉ từ phần train của fold.\"\"\"\n    ratios = []\n    for c in range(4):\n        pos = np.asarray(X[y == c].sum(axis=0)).ravel() + 1\n        neg = np.asarray(X[y != c].sum(axis=0)).ravel() + 1\n        ratios.append(np.log((pos / pos.sum()) / (neg / neg.sum())))\n    return ratios\n","select_v2.py":"\"\"\"Chọn mô hình/ensemble và bias bằng OOF train, không xem nhãn dev.\"\"\"\nimport json\nfrom pathlib import Path\nimport numpy as np\n\n\ndef macro(y, pred):\n    cm = np.bincount(4 * y + pred, minlength=16).reshape(4, 4)\n    den = cm.sum(axis=0) + cm.sum(axis=1)\n    return float(np.mean(np.divide(2 * np.diag(cm), den, out=np.zeros(4), where=den > 0)))\n\n\ndef calibrate(scores, y):\n    # 3 offsets tương đối so với nhãn 0; grid nhỏ để hạn chế overfit.\n    bias = np.zeros(4)\n    for _ in range(3):\n        previous = bias.copy()\n        for cls in [1, 2, 3]:\n            choices = []\n            for value in np.linspace(-.4, .4, 17):\n                proposal = bias.copy()\n                proposal[cls] = value\n                objective = macro(y, (scores + proposal).argmax(axis=1)) - .005 * np.square(proposal).sum()\n                choices.append((objective, -abs(value), float(value)))\n            bias[cls] = max(choices)[2]\n        if np.allclose(previous, bias):\n            break\n    return bias\n\n\ndef select(scores, y, folds, configs):\n    # NB-SVM C=1/2 có cảnh báo chưa hội tụ ở thử nghiệm, không chọn chúng.\n    candidates = [(cfg['name'], {i: 1.}) for i, cfg in enumerate(configs)\n                  if cfg['kind'] != 'nb' or cfg['C'] == .5]\n    # Blends định trước, tránh tìm hàng trăm trọng số trên lớp hiếm.\n    byname = {c['name']: i for i, c in enumerate(configs)}\n    for svm_w in [.25, .5, .75]:\n        candidates.append((f'blend_svm_nb_{svm_w}',\n                           {byname['svm_0.5_balanced']: svm_w, byname['nb_0.5']: 1-svm_w}))\n    rows = []\n    for name, weights in candidates:\n        s = sum(scores[i] * w for i, w in weights.items())\n        cross_pred = np.zeros(len(y), dtype=int)\n        for f in np.unique(folds):\n            fit = folds != f\n            b = calibrate(s[fit], y[fit])\n            cross_pred[~fit] = (s[~fit] + b).argmax(axis=1)\n        raw, calibrated = macro(y, s.argmax(axis=1)), macro(y, cross_pred)\n        # Chọn bias chỉ nếu calibration cải thiện khi kiểm tra chéo.\n        use_bias = calibrated > raw + .002\n        row = {'name': name, 'weights': weights, 'raw_oof_macro_f1': raw,\n               'cross_calibration_macro_f1': calibrated, 'use_bias': use_bias,\n               'selection_score': calibrated if use_bias else raw,\n               'bias': calibrate(s, y).tolist() if use_bias else [0., 0., 0., 0.]}\n        rows.append(row)\n    rows.sort(key=lambda r: r['selection_score'], reverse=True)\n    return rows\n\n\nif __name__ == '__main__':\n    folder = Path('kaggle_ds102/experiments_v2')\n    z = np.load(folder/'oof.npz')\n    cfg = json.loads((folder/'configs.json').read_text())\n    rows = select(z['scores'], z['y'], z['folds'], cfg)\n    (folder/'selection.json').write_text(json.dumps(rows, indent=2))\n    for r in rows:\n        print(r['name'], 'raw', round(r['raw_oof_macro_f1'], 5),\n              'cross calibrated', round(r['cross_calibration_macro_f1'], 5), 'bias', r['bias'])\n","kernel_experiment_colab.py":"\"\"\"Nonlinear SVM experiment. Hosted Colab only, no pretrained models.\n\nTrain/development data remain separate. Run main from a notebook thread to\nallow progress checks through MCP without keeping one tool call open.\n\"\"\"\nimport gc\nimport json\nimport os\nimport platform\nimport traceback\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.metrics.pairwise import rbf_kernel\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.preprocessing import normalize\nfrom sklearn.svm import SVC\nfrom train_submit import load_data, SEED, metrics\nfrom improved_features import prepare, FeaturesV2\nfrom select_v2 import macro\n\n\ndef kernel(text_a, meta_a, text_b, meta_b):\n    k = (text_a @ text_b.T).toarray()\n    k += 1\n    np.square(k, out=k)\n    k += .5 * rbf_kernel(meta_a, meta_b, gamma=.3)\n    k /= 4.5\n    return np.ascontiguousarray(k, dtype=np.float64)\n\n\ndef features(fit, validation):\n    f = FeaturesV2()\n    tx, mx = f.fit_transform(fit)\n    tv, mv = f.transform(validation)\n    # Limit rare metadata outliers; no validation data used to fit scaling.\n    mx.data = np.clip(mx.data, -.75, .75)\n    mv.data = np.clip(mv.data, -.75, .75)\n    return normalize(tx), mx, normalize(tv), mv\n\n\ndef main(data_dir='/content/ds102_data', output_dir='/content/kernel_experiment'):\n    if platform.system() != 'Linux' or not os.environ.get('COLAB_RELEASE_TAG'):\n        raise RuntimeError('Run on hosted Google Colab only; local training is disabled.')\n    output = Path(output_dir)\n    output.mkdir(exist_ok=True)\n    def status(**values):\n        (output/'status.json').write_text(json.dumps(values, ensure_ascii=False, indent=2))\n    try:\n        data = load_data(Path(data_dir))\n        train = prepare(data['train'])\n        y = train.spam_label.to_numpy()\n        configs = [{'C':1.,'weight':'balanced'}, {'C':4.,'weight':'balanced'},\n                   {'C':1.,'weight':'sqrt_balanced'}]\n        oof = np.full((len(configs),len(train)),-1,dtype=int)\n        fold_results = [[] for _ in configs]\n        for fold,(tr,va) in enumerate(StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED).split(train,y,groups=train.comment)):\n            status(stage='features',fold=fold+1,results=fold_results)\n            xt,mt,xv,mv = features(train.iloc[tr],train.iloc[va].drop(columns='spam_label'))\n            status(stage='kernel',fold=fold+1,results=fold_results)\n            kt = kernel(xt,mt,xt,mt)\n            kv = kernel(xv,mv,xt,mt)\n            for i,cfg in enumerate(configs):\n                status(stage='fit',fold=fold+1,config=cfg,results=fold_results)\n                weight = cfg['weight']\n                if weight=='sqrt_balanced':\n                    counts=np.bincount(y[tr],minlength=4)\n                    weight={c:float(np.sqrt(len(tr)/(4*counts[c]))) for c in range(4)}\n                model=SVC(kernel='precomputed',C=cfg['C'],class_weight=weight,\n                          cache_size=512,random_state=SEED)\n                model.fit(kt,y[tr])\n                oof[i,va]=model.predict(kv)\n                fold_results[i].append(macro(y[va],oof[i,va]))\n                del model\n            del kt,kv,xt,mt,xv,mv\n            gc.collect()\n            np.savez_compressed(output/'partial.npz',oof=oof,y=y)\n        scores=[macro(y,p) for p in oof]\n        chosen=int(np.argmax(scores))\n        report={'configs':configs,'fold_scores':fold_results,'oof_macro_f1':scores,\n                'chosen_index':chosen,'fit_data':'train only','pretrained':False}\n        (output/'cv_results.json').write_text(json.dumps(report,indent=2))\n        np.savez_compressed(output/'oof.npz',oof=oof,y=y)\n        status(stage='complete',**report)\n    except BaseException as exc:\n        status(stage='error',error=str(exc),traceback=traceback.format_exc())\n        raise\n\n\nif __name__=='__main__':\n    main()\n"}
SOURCE_DIR=Path('/content/ds102_src'); SOURCE_DIR.mkdir(exist_ok=True)
for name, content in source_files.items():
    (SOURCE_DIR/name).write_text(content,encoding='utf-8')
sys.path.insert(0,str(SOURCE_DIR))
print('READY: experiment source installed on Colab; sklearn', importlib.metadata.version('scikit-learn'))

In [ ]:
import os, threading
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','VECLIB_MAXIMUM_THREADS']:
    os.environ[key]='2'
import kernel_experiment_colab
kernel_job=threading.Thread(target=kernel_experiment_colab.main,daemon=True)
kernel_job.start()
print('Started nonlinear SVM CV on hosted Colab CPU; no pretrained model.')

In [ ]:
from pathlib import Path
import json
for folder in ['kernel_experiment','kernel_lowc','blend_experiment','blend_kernel']:
    p=Path('/content')/folder/'status.json'
    if not p.exists(): print(folder,'not started');continue
    s=json.loads(p.read_text())
    if s.get('stage')=='complete' and 'oof_macro_f1' in s:
        print(folder,'CV:',s['oof_macro_f1'])
    else: print(folder,json.dumps(s,ensure_ascii=False))

In [ ]:
from pathlib import Path
(Path('/content/ds102_src')/'kernel_finalize_colab.py').write_text("\"\"\"Fit the train-CV-selected nonlinear model on hosted Colab only.\"\"\"\nimport gc\nimport hashlib\nimport json\nimport os\nimport platform\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.preprocessing import normalize\nfrom sklearn.svm import SVC\nfrom train_submit import load_data, metrics, SEED\nfrom improved_features import prepare, FeaturesV2\nfrom kernel_experiment_colab import kernel\n\n\ndef main(data_dir='/content/ds102_data',cv_dir='/content/kernel_experiment',output_dir='/content/kernel_final'):\n    if platform.system()!='Linux' or not os.environ.get('COLAB_RELEASE_TAG'):\n        raise RuntimeError('Hosted Colab required; local training disabled.')\n    out=Path(output_dir);out.mkdir(exist_ok=True)\n    def status(**values):\n        (out/'status.json').write_text(json.dumps(values,ensure_ascii=False,indent=2))\n    try:\n        report=json.loads((Path(cv_dir)/'cv_results.json').read_text())\n        best=report['chosen_index']; cfg=report['configs'][best]\n        if report['oof_macro_f1'][best] <= .7651232845255038:\n            status(stage='not_improved',cv=report,reference_v3_oof=.7651232845255038)\n            return\n        data=load_data(Path(data_dir))\n        train,dev,test=[prepare(data[n]) for n in ['train','dev','test']]\n        y=train.spam_label.to_numpy()\n        f=FeaturesV2()\n        status(stage='features',chosen_config=cfg)\n        tx,mx=f.fit_transform(train);tx=normalize(tx)\n        mx.data=np.clip(mx.data,-.75,.75)\n        weight=cfg['weight']\n        if weight=='sqrt_balanced':\n            counts=np.bincount(y,minlength=4)\n            weight={c:float(np.sqrt(len(y)/(4*counts[c]))) for c in range(4)}\n        status(stage='full_kernel',chosen_config=cfg)\n        k=kernel(tx,mx,tx,mx)\n        status(stage='fit',chosen_config=cfg)\n        model=SVC(C=cfg['C'],kernel='precomputed',class_weight=weight,cache_size=512,random_state=SEED)\n        model.fit(k,y)\n        del k;gc.collect()\n        def predict(frame):\n            result=[]\n            for start in range(0,len(frame),512):\n                a,b=f.transform(frame.iloc[start:start+512])\n                a=normalize(a);b.data=np.clip(b.data,-.75,.75)\n                kk=kernel(a,b,tx,mx)\n                result.extend(model.predict(kk).tolist())\n            return np.asarray(result,dtype=int)\n        status(stage='predict_dev',chosen_config=cfg)\n        dev_pred=predict(dev)\n        status(stage='predict_test',chosen_config=cfg)\n        test_pred=predict(test)\n        submission=data['samplesubmission'].copy()\n        by_id=dict(zip(test.id.astype(str),test_pred.tolist()))\n        submission['spam_label']=submission.ID.astype(str).map(by_id)\n        assert not submission.isna().any().any()\n        assert submission.spam_label.isin([0,1,2,3]).all()\n        submission['spam_label']=submission.spam_label.astype(int)\n        path=out/'submission_kernel.csv';submission.to_csv(path,index=False)\n        import pandas as pd\n        saved=pd.read_csv(path)\n        pd.testing.assert_frame_equal(saved,submission,check_dtype=False)\n        assert len(saved)==len(test) and saved.ID.equals(data['samplesubmission'].ID)\n        unseen=~dev.comment.isin(set(train.comment)).to_numpy()\n        result={'cv':report,'chosen_config':cfg,'dev':metrics(dev.spam_label.to_numpy(),dev_pred),\n                'dev_without_comment_overlap':metrics(dev.spam_label.to_numpy()[unseen],dev_pred[unseen]),\n                'rows_verified':len(saved),'pretrained':False,'execution':'hosted Google Colab CPU',\n                'fit_data':'train only; dev/test not used to fit features/model or choose parameters',\n                'data_sha256':{n:hashlib.sha256((Path(data_dir)/f'ds102_{n}.csv').read_bytes()).hexdigest() for n in ['train','dev','test']}}\n        (out/'metrics.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))\n        status(stage='complete',result=result)\n    except BaseException as exc:\n        import traceback\n        status(stage='error',error=str(exc),traceback=traceback.format_exc())\n        raise\n\n\nif __name__=='__main__':\n    main()\n",encoding='utf-8')
print('Final-fit source ready; no training started by this cell.')

In [ ]:
from pathlib import Path
(Path('/content/ds102_src')/'kernel_experiment_colab.py').write_text("\"\"\"Nonlinear SVM experiment. Hosted Colab only, no pretrained models.\n\nTrain/development data remain separate. Run main from a notebook thread to\nallow progress checks through MCP without keeping one tool call open.\n\"\"\"\nimport gc\nimport json\nimport os\nimport platform\nimport traceback\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.metrics.pairwise import rbf_kernel\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.preprocessing import normalize\nfrom sklearn.svm import SVC\nfrom train_submit import load_data, SEED, metrics\nfrom improved_features import prepare, FeaturesV2\nfrom select_v2 import macro\n\n\ndef kernel(text_a, meta_a, text_b, meta_b):\n    k = (text_a @ text_b.T).toarray()\n    k += 1\n    np.square(k, out=k)\n    k += .5 * rbf_kernel(meta_a, meta_b, gamma=.3)\n    k /= 4.5\n    return np.ascontiguousarray(k, dtype=np.float64)\n\n\ndef features(fit, validation):\n    f = FeaturesV2()\n    tx, mx = f.fit_transform(fit)\n    tv, mv = f.transform(validation)\n    # Limit rare metadata outliers; no validation data used to fit scaling.\n    mx.data = np.clip(mx.data, -.75, .75)\n    mv.data = np.clip(mv.data, -.75, .75)\n    return normalize(tx), mx, normalize(tv), mv\n\n\ndef main(data_dir='/content/ds102_data', output_dir='/content/kernel_experiment', configs_override=None):\n    if platform.system() != 'Linux' or not os.environ.get('COLAB_RELEASE_TAG'):\n        raise RuntimeError('Run on hosted Google Colab only; local training is disabled.')\n    output = Path(output_dir)\n    output.mkdir(exist_ok=True)\n    def status(**values):\n        (output/'status.json').write_text(json.dumps(values, ensure_ascii=False, indent=2))\n    try:\n        data = load_data(Path(data_dir))\n        train = prepare(data['train'])\n        y = train.spam_label.to_numpy()\n        configs = configs_override or [{'C':1.,'weight':'balanced'}, {'C':4.,'weight':'balanced'},\n                                      {'C':1.,'weight':'sqrt_balanced'}]\n        oof = np.full((len(configs),len(train)),-1,dtype=int)\n        fold_results = [[] for _ in configs]\n        for fold,(tr,va) in enumerate(StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED).split(train,y,groups=train.comment)):\n            status(stage='features',fold=fold+1,results=fold_results)\n            xt,mt,xv,mv = features(train.iloc[tr],train.iloc[va].drop(columns='spam_label'))\n            status(stage='kernel',fold=fold+1,results=fold_results)\n            kt = kernel(xt,mt,xt,mt)\n            kv = kernel(xv,mv,xt,mt)\n            for i,cfg in enumerate(configs):\n                status(stage='fit',fold=fold+1,config=cfg,results=fold_results)\n                weight = cfg['weight']\n                if weight=='sqrt_balanced':\n                    counts=np.bincount(y[tr],minlength=4)\n                    weight={c:float(np.sqrt(len(tr)/(4*counts[c]))) for c in range(4)}\n                model=SVC(kernel='precomputed',C=cfg['C'],class_weight=weight,\n                          cache_size=512,random_state=SEED)\n                model.fit(kt,y[tr])\n                oof[i,va]=model.predict(kv)\n                fold_results[i].append(macro(y[va],oof[i,va]))\n                del model\n            del kt,kv,xt,mt,xv,mv\n            gc.collect()\n            np.savez_compressed(output/'partial.npz',oof=oof,y=y)\n        scores=[macro(y,p) for p in oof]\n        chosen=int(np.argmax(scores))\n        report={'configs':configs,'fold_scores':fold_results,'oof_macro_f1':scores,\n                'chosen_index':chosen,'fit_data':'train only','pretrained':False}\n        (output/'cv_results.json').write_text(json.dumps(report,indent=2))\n        np.savez_compressed(output/'oof.npz',oof=oof,y=y)\n        status(stage='complete',**report)\n    except BaseException as exc:\n        status(stage='error',error=str(exc),traceback=traceback.format_exc())\n        raise\n\n\nif __name__=='__main__':\n    main()\n",encoding='utf-8')
import importlib, threading
def low_c_followup():
    kernel_job.join()
    importlib.reload(kernel_experiment_colab)
    configs=[{'C':c,'weight':w} for c in [.1,.25] for w in ['balanced','sqrt_balanced']]
    kernel_experiment_colab.main(output_dir='/content/kernel_lowc',configs_override=configs)
lowc_job=threading.Thread(target=low_c_followup,daemon=True)
lowc_job.start()
print('Follow-up queued on Colab only; no pretrained model.')

In [ ]:
import hashlib
from pathlib import Path
expected={"train":"15202b19b2dc2c6c89afeb209024bb0dc77ddf927de44381a96c24a961d823a5","dev":"d895cc298b1e5b56bbb95c9f9b15610e5207a00cfd12cc79f32061e27b7a8ccf","test":"82c21afd6f8f4610912bc12ed19c3b6b1a586fb0dfa5d2296f6548ae4fc6b398"}
for name, expected_hash in expected.items():
    actual=hashlib.sha256((Path('/content/ds102_data')/f'ds102_{name}.csv').read_bytes()).hexdigest()
    assert actual==expected_hash, f'Data mismatch: {name}'
print('PASS: Colab train/dev/test match the original data byte for byte.')

In [ ]:
import threading, json, traceback
from pathlib import Path
def finish_after_cv():
    try:
        lowc_job.join()
        import kernel_finalize_colab
        candidates=[Path('/content/kernel_experiment'),Path('/content/kernel_lowc')]
        candidates=[p for p in candidates if (p/'cv_results.json').is_file()]
        best=max(candidates,key=lambda p:max(json.loads((p/'cv_results.json').read_text())['oof_macro_f1']))
        kernel_finalize_colab.main(cv_dir=str(best))
    except BaseException as e:
        p=Path('/content/kernel_final');p.mkdir(exist_ok=True)
        (p/'status.json').write_text(json.dumps({'stage':'error','error':str(e),'traceback':traceback.format_exc()}))
final_job=threading.Thread(target=finish_after_cv,daemon=True)
final_job.start()
print('Final fitting queued only if selected train CV exceeds v3 reference.')

In [ ]:
import sys, subprocess, importlib.util
if importlib.util.find_spec('catboost') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','catboost==1.2.10'])
from pathlib import Path
source_files={"train_submit_v2.py":"\"\"\"DS102 v2: đặc trưng quan hệ review/sản phẩm, SVM và calibration F1-macro.\"\"\"\nimport argparse\nimport hashlib\nimport json\nimport platform\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom scipy import sparse\nfrom sklearn.metrics import classification_report\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.svm import LinearSVC\nfrom train_submit import load_data, find_data_dir, make_model, make_features, metrics, LABELS, SEED\nfrom improved_features import prepare, FeaturesV2, nb_ratios\nfrom select_v2 import macro, calibrate\n\n\n# Cấu hình được chốt từ cross-validation trên train, trước đánh giá dev v2.\nRECIPE = [\n    {'kind': 'svm', 'C': .5, 'weight': 'balanced', 'ensemble_weight': .75},\n    {'kind': 'nb', 'C': .5, 'weight': 'balanced', 'ensemble_weight': .25},\n]\n\n\ndef fit_predict(fit_df, predict_frames, recipe):\n    \"\"\"Không nhận nhãn của các tập dự đoán.\"\"\"\n    y = fit_df.spam_label.to_numpy()\n    features = FeaturesV2()\n    Xt, Mt = features.fit_transform(fit_df)\n    transformed = [features.transform(df) for df in predict_frames]\n    result = [np.zeros((len(df), 4)) for df in predict_frames]\n    ratios = nb_ratios(Xt, y) if any(c['kind'] == 'nb' for c in recipe) else None\n    for config in recipe:\n        if config['kind'] == 'svm':\n            model = make_model(config, y).fit(sparse.hstack([Xt, Mt], format='csr'), y)\n            if np.max(model.n_iter_) >= model.max_iter:\n                raise RuntimeError('SVM chưa hội tụ; tăng max_iter trước khi xuất kết quả.')\n            for j, (Xv, Mv) in enumerate(transformed):\n                result[j] += config['ensemble_weight'] * model.decision_function(sparse.hstack([Xv, Mv], format='csr'))\n        elif config['kind'] == 'nb':\n            for cls, r in enumerate(ratios):\n                model = LinearSVC(C=config['C'], class_weight='balanced', dual=True,\n                                  max_iter=30000, random_state=SEED)\n                model.fit(sparse.hstack([Xt.multiply(r), Mt], format='csr'), (y == cls).astype(int))\n                if np.max(model.n_iter_) >= model.max_iter:\n                    raise RuntimeError('NB-SVM chưa hội tụ; tăng max_iter trước khi xuất kết quả.')\n                for j, (Xv, Mv) in enumerate(transformed):\n                    result[j][:, cls] += config['ensemble_weight'] * model.decision_function(\n                        sparse.hstack([Xv.multiply(r), Mv], format='csr'))\n        else:\n            raise ValueError(f\"Model không được hỗ trợ: {config['kind']}\")\n    return result\n\n\ndef main_v2(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--data-dir', default=None)\n    parser.add_argument('--output-dir', default='output_v2')\n    parser.add_argument('--folds', type=int, default=3)\n    args = parser.parse_args(argv)\n    if args.folds < 2:\n        parser.error('--folds phải >= 2')\n    data_dir = find_data_dir(args.data_dir)\n    data = load_data(data_dir)\n    expected = {'comment', 'product_name', 'description', 'category', 'categories', 'rating', 'num_sold', 'num_review'}\n    for name in ['train', 'dev', 'test']:\n        missing = expected - set(data[name].columns)\n        if missing:\n            raise ValueError(f'{name} thiếu cột {missing}')\n        for col in ['rating', 'num_sold', 'num_review']:\n            values = pd.to_numeric(data[name][col], errors='raise')\n            if not np.isfinite(values).all() or (values < 0).any():\n                raise ValueError(f'{name}.{col}: giá trị số không hợp lệ')\n            data[name][col] = values\n    train, dev, test = [prepare(data[n]) for n in ['train', 'dev', 'test']]\n    output = Path(args.output_dir).expanduser().resolve()\n    output.mkdir(parents=True, exist_ok=True)\n    y = train.spam_label.to_numpy()\n    oof = np.zeros((len(train), 4))\n    fold_ids = np.full(len(train), -1)\n    splitter = StratifiedGroupKFold(n_splits=args.folds, shuffle=True, random_state=SEED)\n    for fold, (tr, va) in enumerate(splitter.split(train, y, groups=train.comment)):\n        if set(y[tr]) != set(LABELS):\n            raise ValueError('Fold không đủ 4 lớp; giảm số folds.')\n        print(f'CV fold {fold+1}/{args.folds}', flush=True)\n        oof[va] = fit_predict(train.iloc[tr], [train.iloc[va].drop(columns='spam_label')], RECIPE)[0]\n        fold_ids[va] = fold\n        print(f'  Raw F1-macro={macro(y[va], oof[va].argmax(axis=1)):.5f}', flush=True)\n    cross_pred = np.zeros(len(train), dtype=int)\n    for f in range(args.folds):\n        fit = fold_ids != f\n        bias_fold = calibrate(oof[fit], y[fit])\n        cross_pred[~fit] = (oof[~fit] + bias_fold).argmax(axis=1)\n    raw_score = macro(y, oof.argmax(axis=1))\n    cross_score = macro(y, cross_pred)\n    use_bias = cross_score > raw_score + .002\n    bias = calibrate(oof, y) if use_bias else np.zeros(4)\n    print(f'OOF raw={raw_score:.5f}; cross-calibrated={cross_score:.5f}; bias={bias.tolist()}', flush=True)\n    print('Fit toàn bộ train và dự đoán dev/test...', flush=True)\n    dev_scores, test_scores = fit_predict(train, [dev.drop(columns='spam_label'), test], RECIPE)\n    dev_pred = (dev_scores + bias).argmax(axis=1)\n    test_pred = (test_scores + bias).argmax(axis=1)\n    dev_y = dev.spam_label.to_numpy()\n    unique_dev = ~dev.comment.isin(set(train.comment)).to_numpy()\n    dev_report = metrics(dev_y, dev_pred)\n    print(f\"Dev F1-macro={dev_report['macro_f1']:.5f}\", flush=True)\n    print(classification_report(dev_y, dev_pred, labels=LABELS, digits=4, zero_division=0), flush=True)\n    by_id = pd.Series(test_pred, index=test.id.astype(str))\n    submission = data['samplesubmission'].copy()\n    submission['spam_label'] = submission.ID.astype(str).map(by_id)\n    if submission.spam_label.isna().any() or not submission.spam_label.isin(LABELS).all():\n        raise ValueError('Nhãn submission không hợp lệ')\n    submission['spam_label'] = submission.spam_label.astype(int)\n    path = output/'submission_v2.csv'\n    submission.to_csv(path, index=False)\n    saved = pd.read_csv(path)\n    pd.testing.assert_frame_equal(saved, submission, check_dtype=False)\n    if len(saved) != len(test) or not saved.ID.equals(data['samplesubmission'].ID):\n        raise ValueError('Sai số dòng hoặc thứ tự ID')\n    pd.DataFrame({'comment': data['dev'].comment, 'true_label': dev_y,\n                  'predicted_label': dev_pred, 'comment_seen_in_train': ~unique_dev}).to_csv(output/'dev_predictions_v2.csv', index=False)\n    np.savez_compressed(output/'scores_v2.npz', oof=oof, y=y, folds=fold_ids,\n                        dev_scores=dev_scores, test_scores=test_scores, bias=bias)\n    report = {\n        'recipe': RECIPE, 'seed': SEED, 'folds': args.folds, 'python': platform.python_version(),\n        'sklearn': sklearn.__version__, 'data_sha256': {name: hashlib.sha256((data_dir/file).read_bytes()).hexdigest()\n            for name, file in [('train','ds102_train.csv'), ('dev','ds102_dev.csv'), ('test','ds102_test.csv')]},\n        'fit_data': 'train only; no dev or test used to fit features, models, or calibration',\n        'oof_raw_macro_f1': raw_score, 'oof_cross_calibration_macro_f1': cross_score,\n        'calibration_bias': bias.tolist(), 'dev': dev_report,\n        'dev_without_comment_overlap': metrics(dev_y[unique_dev], dev_pred[unique_dev]) if unique_dev.any() else None,\n        'selection_caveat': 'CV estimates were used for model selection; dev was previously used to inspect baseline errors.',\n        'submission_rows_verified': len(saved),\n        'prediction_counts': {int(k): int(v) for k, v in saved.spam_label.value_counts().items()},\n    }\n    (output/'metrics_v2.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')\n    print(f'Đã kiểm tra và lưu {len(saved)} dòng: {path}', flush=True)\n    return report\n\n\nif __name__ == '__main__':\n    main_v2()\n","catboost_v3.py":"\"\"\"CatBoost text + thông tin review/sản phẩm, không dùng mô hình pretrained.\"\"\"\nimport numpy as np\nimport pandas as pd\nfrom catboost import CatBoostClassifier\nfrom improved_features import dense_features, tokens\nfrom train_submit import SEED\n\n\ndef cat_features(df):\n    a = dense_features(df)\n    x = pd.DataFrame(a, columns=[f'stat_{i}' for i in range(a.shape[1])])\n    x['comment'] = [' '.join(tokens(t)) or 'emptyreview' for t in df.comment]\n    x['category'] = df.category.astype(str).to_numpy()\n    x['rating'] = df.rating.astype(str).to_numpy()\n    return x\n\n\ndef make_cat(weight='SqrtBalanced', iterations=500):\n    return CatBoostClassifier(\n        iterations=iterations, depth=5, learning_rate=.1, l2_leaf_reg=5,\n        loss_function='MultiClass', auto_class_weights=weight,\n        thread_count=4, random_seed=SEED, allow_writing_files=False,\n        text_features=['comment'], cat_features=['category', 'rating'],\n        tokenizers=[{'tokenizer_id':'Space', 'separator_type':'ByDelimiter', 'delimiter':' '}],\n        dictionaries=[{'dictionary_id':'Word', 'gram_order':'1', 'occurrence_lower_bound':'3', 'max_dictionary_size':'40000'},\n                      {'dictionary_id':'Bigram', 'gram_order':'2', 'occurrence_lower_bound':'3', 'max_dictionary_size':'40000'}],\n        feature_calcers=['BoW:top_tokens_count=700', 'NaiveBayes', 'BM25'],\n        verbose=False,\n    )\n","kernel_finalize_colab.py":"\"\"\"Fit the train-CV-selected nonlinear model on hosted Colab only.\"\"\"\nimport gc\nimport hashlib\nimport json\nimport os\nimport platform\nfrom pathlib import Path\nimport numpy as np\nfrom sklearn.preprocessing import normalize\nfrom sklearn.svm import SVC\nfrom train_submit import load_data, metrics, SEED\nfrom improved_features import prepare, FeaturesV2\nfrom kernel_experiment_colab import kernel\n\n\ndef main(data_dir='/content/ds102_data',cv_dir='/content/kernel_experiment',output_dir='/content/kernel_final',allow_cv_lower_for_blend=False):\n    if platform.system()!='Linux' or not os.environ.get('COLAB_RELEASE_TAG'):\n        raise RuntimeError('Hosted Colab required; local training disabled.')\n    out=Path(output_dir);out.mkdir(exist_ok=True)\n    def status(**values):\n        (out/'status.json').write_text(json.dumps(values,ensure_ascii=False,indent=2))\n    try:\n        report=json.loads((Path(cv_dir)/'cv_results.json').read_text())\n        best=report['chosen_index']; cfg=report['configs'][best]\n        if report['oof_macro_f1'][best] <= .7651232845255038 and not allow_cv_lower_for_blend:\n            status(stage='not_improved',cv=report,reference_v3_oof=.7651232845255038)\n            return\n        data=load_data(Path(data_dir))\n        train,dev,test=[prepare(data[n]) for n in ['train','dev','test']]\n        y=train.spam_label.to_numpy()\n        f=FeaturesV2()\n        status(stage='features',chosen_config=cfg)\n        tx,mx=f.fit_transform(train);tx=normalize(tx)\n        mx.data=np.clip(mx.data,-.75,.75)\n        weight=cfg['weight']\n        if weight=='sqrt_balanced':\n            counts=np.bincount(y,minlength=4)\n            weight={c:float(np.sqrt(len(y)/(4*counts[c]))) for c in range(4)}\n        status(stage='full_kernel',chosen_config=cfg)\n        k=kernel(tx,mx,tx,mx)\n        status(stage='fit',chosen_config=cfg)\n        model=SVC(C=cfg['C'],kernel='precomputed',class_weight=weight,cache_size=512,random_state=SEED)\n        model.fit(k,y)\n        del k;gc.collect()\n        def predict(frame):\n            result=[]\n            for start in range(0,len(frame),512):\n                a,b=f.transform(frame.iloc[start:start+512])\n                a=normalize(a);b.data=np.clip(b.data,-.75,.75)\n                kk=kernel(a,b,tx,mx)\n                result.extend(model.predict(kk).tolist())\n            return np.asarray(result,dtype=int)\n        status(stage='predict_dev',chosen_config=cfg)\n        dev_pred=predict(dev)\n        status(stage='predict_test',chosen_config=cfg)\n        test_pred=predict(test)\n        submission=data['samplesubmission'].copy()\n        by_id=dict(zip(test.id.astype(str),test_pred.tolist()))\n        submission['spam_label']=submission.ID.astype(str).map(by_id)\n        assert not submission.isna().any().any()\n        assert submission.spam_label.isin([0,1,2,3]).all()\n        submission['spam_label']=submission.spam_label.astype(int)\n        path=out/'submission_kernel.csv';submission.to_csv(path,index=False)\n        import pandas as pd\n        saved=pd.read_csv(path)\n        np.savez_compressed(out/'predictions.npz',dev=dev_pred,test=test_pred)\n        pd.testing.assert_frame_equal(saved,submission,check_dtype=False)\n        assert len(saved)==len(test) and saved.ID.equals(data['samplesubmission'].ID)\n        unseen=~dev.comment.isin(set(train.comment)).to_numpy()\n        result={'cv':report,'chosen_config':cfg,'dev':metrics(dev.spam_label.to_numpy(),dev_pred),\n                'dev_without_comment_overlap':metrics(dev.spam_label.to_numpy()[unseen],dev_pred[unseen]),\n                'rows_verified':len(saved),'pretrained':False,'execution':'hosted Google Colab CPU',\n                'fit_data':'train only; dev/test not used to fit features/model or choose parameters',\n                'data_sha256':{n:hashlib.sha256((Path(data_dir)/f'ds102_{n}.csv').read_bytes()).hexdigest() for n in ['train','dev','test']}}\n        (out/'metrics.json').write_text(json.dumps(result,ensure_ascii=False,indent=2))\n        status(stage='complete',result=result)\n    except BaseException as exc:\n        import traceback\n        status(stage='error',error=str(exc),traceback=traceback.format_exc())\n        raise\n\n\nif __name__=='__main__':\n    main()\n","cloud_blend_experiment.py":"\"\"\"Recompute reference CV on Colab; never uploads local cached predictions.\"\"\"\nimport gc,json,os,platform,traceback\nfrom pathlib import Path\nimport numpy as np\nfrom scipy.special import softmax\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom train_submit import load_data,SEED,metrics\nfrom improved_features import prepare\nfrom train_submit_v2 import fit_predict,RECIPE\nfrom catboost_v3 import make_cat,cat_features\nfrom select_v2 import macro\n\n\ndef v3_scores(fit, frames):\n    scores=fit_predict(fit,frames,RECIPE)\n    model=make_cat('SqrtBalanced',iterations=500)\n    model.fit(cat_features(fit),fit.spam_label.to_numpy(),verbose=False)\n    return [.5*softmax(3*s,axis=1)+.5*model.predict_proba(cat_features(d)) for s,d in zip(scores,frames)]\n\n\ndef main():\n    if platform.system()!='Linux' or not os.environ.get('COLAB_RELEASE_TAG'):\n        raise RuntimeError('Hosted Colab only; no local fitting.')\n    out=Path('/content/blend_experiment');out.mkdir(exist_ok=True)\n    def status(**d):(out/'status.json').write_text(json.dumps(d,ensure_ascii=False,indent=2))\n    try:\n        data=load_data(Path('/content/ds102_data'))\n        train,dev,test=[prepare(data[n]) for n in ['train','dev','test']]\n        y=train.spam_label.to_numpy();p=np.zeros((len(y),4));fold_ids=np.full(len(y),-1)\n        dirs=[Path('/content/kernel_experiment'),Path('/content/kernel_lowc')]\n        best_dir=max(dirs,key=lambda d:max(json.loads((d/'cv_results.json').read_text())['oof_macro_f1']))\n        kr=json.loads((best_dir/'cv_results.json').read_text());kz=np.load(best_dir/'oof.npz')\n        assert np.array_equal(kz['y'],y)\n        kp=kz['oof'][kr['chosen_index']]\n        for f,(tr,va) in enumerate(StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=SEED).split(train,y,groups=train.comment)):\n            status(stage='v3_reference_CV',fold=f+1)\n            p[va]=v3_scores(train.iloc[tr],[train.iloc[va].drop(columns='spam_label')])[0]\n            fold_ids[va]=f;gc.collect()\n        results=[]\n        for a in [0.,.05,.1,.2]:\n            pred=((1-a)*p+a*np.eye(4)[kp]).argmax(1)\n            results.append({'kernel_weight':a,'oof_macro_f1':macro(y,pred),\n                            'fold_macro_f1':[macro(y[fold_ids==f],pred[fold_ids==f]) for f in range(3)]})\n        np.savez_compressed(out/'oof.npz',v3_scores=p,y=y,folds=fold_ids,kernel_pred=kp)\n        (out/'cv_results.json').write_text(json.dumps(results,indent=2))\n        best=max(results,key=lambda d:d['oof_macro_f1']);base=results[0]['oof_macro_f1']\n        if best['kernel_weight']==0 or best['oof_macro_f1'] < base+.002:\n            status(stage='not_improved',results=results,kernel_cv=kr)\n            return\n        status(stage='fit_final_v3',results=results,chosen=best)\n        dev_p,test_p=v3_scores(train,[dev.drop(columns='spam_label'),test]);gc.collect()\n        status(stage='fit_final_kernel',results=results,chosen=best)\n        import kernel_finalize_colab\n        kernel_finalize_colab.main(cv_dir=str(best_dir),output_dir='/content/blend_kernel',allow_cv_lower_for_blend=True)\n        kz=np.load('/content/blend_kernel/predictions.npz')\n        a=best['kernel_weight']\n        dev_pred=((1-a)*dev_p+a*np.eye(4)[kz['dev']]).argmax(1)\n        test_pred=((1-a)*test_p+a*np.eye(4)[kz['test']]).argmax(1)\n        sub=data['samplesubmission'].copy()\n        by_id=dict(zip(test.id.astype(str),test_pred.tolist()))\n        sub['spam_label']=sub.ID.astype(str).map(by_id)\n        assert not sub.isna().any().any() and sub.spam_label.isin([0,1,2,3]).all()\n        sub['spam_label']=sub.spam_label.astype(int)\n        path=out/'submission_v4.csv';sub.to_csv(path,index=False)\n        import pandas as pd\n        pd.testing.assert_frame_equal(pd.read_csv(path),sub,check_dtype=False)\n        unseen=~dev.comment.isin(set(train.comment)).to_numpy()\n        report={'selection':results,'chosen':best,'kernel_cv':kr,\n                'v3_dev':metrics(dev.spam_label.to_numpy(),dev_p.argmax(1)),\n                'dev':metrics(dev.spam_label.to_numpy(),dev_pred),\n                'dev_without_overlap':metrics(dev.spam_label.to_numpy()[unseen],dev_pred[unseen]),\n                'rows_verified':len(sub),'pretrained':False,'fit_data':'train only','execution':'Google Colab CPU',\n                'caveat':'CV used for model selection; dev reused in iterative development; new Kaggle score unknown'}\n        (out/'metrics.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))\n        status(stage='complete',report=report)\n    except BaseException as e:\n        status(stage='error',error=str(e),traceback=traceback.format_exc());raise\n\n\nif __name__=='__main__':main()\n"}
for name,content in source_files.items():
    (Path('/content/ds102_src')/name).write_text(content,encoding='utf-8')
print('Source ready for fresh Colab training; no local cached predictions uploaded.')

In [ ]:
import threading, importlib
final_job.join()
import cloud_blend_experiment
blend_job=threading.Thread(target=cloud_blend_experiment.main,daemon=True)
blend_job.start()
print('Recomputing v3 reference predictions on Colab from the already uploaded CSV files.')

In [ ]:
import importlib, kernel_finalize_colab
importlib.reload(kernel_finalize_colab)
print('Final-fit module refreshed; no fitting started by this cell.')

In [ ]:
import json
from pathlib import Path
blend_job.join()
status=json.loads(Path('/content/blend_experiment/status.json').read_text())
print(json.dumps(status,ensure_ascii=False,indent=2))
path=Path('/content/blend_experiment/submission_v4.csv')
if status.get('stage')=='complete' and path.is_file():
    from google.colab import files
    files.download(str(path))
elif status.get('stage')=='not_improved':
    print('No candidate beat v3 reliably in train CV; keep submission_v3.csv.')